# Research Agents (A2A)

Explore the Researcher and Writer agents that form the execution core of the harness.
These agents are exposed as A2A-compliant services and called by the harness controller.
These agents are the execution backbone of the AGENTS.md harness — called during the Execute phase of each iteration.

## Learning Objectives
- Understand the role of execution agents in the harness
- Examine the A2A protocol integration
- See how agents receive tasks from the orchestrator

In [ ]:
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

print("Environment loaded")

## 1. Researcher Agent

The Researcher agent performs:
- Query rewriting for better retrieval
- Semantic search across document stores
- Context synthesis from multiple sources

In the harness architecture, these capabilities are exposed via the Tool Layer (MCP servers), and the harness controller invokes them directly. The Researcher agent becomes a thin A2A wrapper for external access.

In [ ]:
# The harness controller uses tool layer functions directly
from agents.orchestrator.layers.tools import semantic_search, rewrite_query, synthesize_context

# Simulate what the researcher does in a single iteration
query = "What are the benefits of using pgvector for RAG applications?"

# Step 1: Rewrite query
queries = rewrite_query(query)
print(f"Original: {query}")
print(f"Rewritten into {len(queries)} variants:")
for q in queries:
    print(f"  - {q}")

In [ ]:
# Step 2: Search with each query variant
all_results = []
for q in queries[:2]:  # Limit for demo
    results = semantic_search(q, top_k=3)
    all_results.extend(results)
    print(f"Query '{q[:50]}...' → {len(results)} results")

# Deduplicate
seen = set()
unique = []
for r in all_results:
    key = (r.get('document_id'), r.get('chunk_index'))
    if key not in seen:
        seen.add(key)
        unique.append(r)

print(f"\nTotal unique results: {len(unique)}")

In [ ]:
# Step 3: Synthesize context
if unique:
    synthesis = synthesize_context(query, unique[:5])
    print("Synthesized context:")
    print(synthesis['synthesis'][:600])
else:
    print("No results to synthesize. Please ingest documents first.")

## 2. Writer Agent

The Writer agent takes accumulated context and produces a structured research report.
In the harness, this is handled by the `draft_report` tool.

In [ ]:
from agents.orchestrator.layers.tools import draft_report
import json

# Simulate report generation
context = synthesis.get('synthesis', 'No context available') if unique else 'Sample context about pgvector.'
plan = json.dumps([{"action": "search", "query": query, "purpose": "Find benefits of pgvector"}])

result = draft_report(query=query, context=context, plan=plan)
print(f"Draft report ({result.get('tokens_used', 0)} tokens):")
print("=" * 60)
print(result['draft'][:1000])

## 3. A2A Protocol

Both agents are also accessible via the A2A (Agent-to-Agent) protocol:

```json
POST http://researcher:8102/
{
  "jsonrpc": "2.0",
  "method": "message/send",
  "params": {
    "message": {
      "role": "user",
      "parts": [{"kind": "text", "text": "Research query here"}]
    }
  }
}
```

The AgentCard at `/.well-known/agent-card.json` describes their capabilities.

## Summary

The execution agents:
- **Researcher**: Multi-query search + context synthesis
- **Writer**: Structured report generation from context

In the AGENTS.md harness architecture, these execution agents are invoked as tool layer operations within the iterative plan-execute-verify-reflect loop.